In [14]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA, NMF
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (classification_report, roc_auc_score, roc_curve,
                              silhouette_score, confusion_matrix, mean_squared_error)
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from scipy.stats import pearsonr, chi2_contingency, f_oneway
import warnings, os
warnings.filterwarnings('ignore')
 
os.makedirs('figures', exist_ok=True)
DATASET_PATH = 'OnlineRetail.xlsx' 

Data loading and cleaning

In [18]:
df_raw = pd.read_excel(DATASET_PATH)
print(f"Raw shape          : {df_raw.shape}")
print(f"Missing CustomerID : {df_raw['CustomerID'].isna().sum()}")
 
# Remove rows without CustomerID, negative quantities (returns), and zero prices
df = df_raw.dropna(subset=['CustomerID']).copy()
df = df[df['Quantity'] > 0]
df = df[df['UnitPrice'] > 0]
df['CustomerID'] = df['CustomerID'].astype(int)
df['Revenue'] = df['Quantity'] * df['UnitPrice']
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['Month'] = df['InvoiceDate'].dt.to_period('M')
df['YearMonth'] = df['InvoiceDate'].dt.strftime('%Y-%m')
df['DayOfWeek'] = df['InvoiceDate'].dt.day_name()
df['Hour'] = df['InvoiceDate'].dt.hour
 
print(f"Clean shape        : {df.shape}")
print(f"Date range         : {df['InvoiceDate'].min()} → {df['InvoiceDate'].max()}")
print(f"Unique customers   : {df['CustomerID'].nunique()}")
print(f"Unique countries   : {df['Country'].nunique()}")
print(f"Total revenue      : £{df['Revenue'].sum():,.2f}")

Raw shape          : (541909, 8)
Missing CustomerID : 135080
Clean shape        : (397884, 13)
Date range         : 2010-12-01 08:26:00 → 2011-12-09 12:50:00
Unique customers   : 4338
Unique countries   : 37
Total revenue      : £8,911,407.90


RFM FEATURE ENGINEERING

In [19]:
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)
rfm = df.groupby('CustomerID').agg(
    Recency   = ('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency = ('InvoiceNo', 'nunique'),
    Monetary  = ('Revenue', 'sum')
).reset_index()
 
print("\nRFM Summary Statistics:")
print(rfm[['Recency', 'Frequency', 'Monetary']].describe().round(2))
 
# RFM Scoring (1–4)
rfm['R_Score'] = pd.qcut(rfm['Recency'], 4, labels=[4, 3, 2, 1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 4, labels=[1, 2, 3, 4]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'], 4, labels=[1, 2, 3, 4]).astype(int)
rfm['RFM_Score'] = rfm['R_Score'] + rfm['F_Score'] + rfm['M_Score']
 
def rfm_segment(score):
    if score >= 10: return 'Champions'
    elif score >= 8: return 'Loyal Customers'
    elif score >= 6: return 'Potential Loyalists'
    elif score >= 4: return 'At Risk'
    else:            return 'Lost'
 
rfm['Segment'] = rfm['RFM_Score'].apply(rfm_segment)
rfm['AOV'] = rfm['Monetary'] / rfm['Frequency']
rfm['CLV_Estimated'] = rfm['AOV'] * rfm['Frequency'] * 2   # 2-year projection
 
print("\nCustomer Segment Distribution:")
print(rfm['Segment'].value_counts())


RFM Summary Statistics:
       Recency  Frequency   Monetary
count  4338.00    4338.00    4338.00
mean     92.54       4.27    2054.27
std     100.01       7.70    8989.23
min       1.00       1.00       3.75
25%      18.00       1.00     307.41
50%      51.00       2.00     674.48
75%     142.00       5.00    1661.74
max     374.00     209.00  280206.02

Customer Segment Distribution:
Segment
Champions              1268
At Risk                 988
Potential Loyalists     936
Loyal Customers         843
Lost                    303
Name: count, dtype: int64


K-MEANS CLUSTERING

In [20]:
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[['Recency', 'Frequency', 'Monetary']])
 
inertias, sil_scores = [], []
K_range = range(2, 10)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(rfm_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(rfm_scaled, km.labels_))
 
optimal_k = 4
km_final = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
rfm['Cluster'] = km_final.fit_predict(rfm_scaled)
final_sil = silhouette_score(rfm_scaled, rfm['Cluster'])
 
cluster_names = {0: 'High Value', 1: 'Mid-Tier', 2: 'VIP/Bulk', 3: 'Inactive'}
rfm['ClusterName'] = rfm['Cluster'].map(cluster_names)
 
print(f"\nOptimal k            : {optimal_k}")
print(f"Silhouette Score     : {final_sil:.4f}")
print("\nCluster Profile (Mean RFM):")
cluster_profile = rfm.groupby('Cluster')[['Recency', 'Frequency', 'Monetary']].mean()
print(cluster_profile.round(2))
 
# PCA for 2D visualization
pca = PCA(n_components=2)
pca_coords = pca.fit_transform(rfm_scaled)
rfm['PC1'] = pca_coords[:, 0]
rfm['PC2'] = pca_coords[:, 1]


Optimal k            : 4
Silhouette Score     : 0.6162

Cluster Profile (Mean RFM):
         Recency  Frequency   Monetary
Cluster                               
0          43.70       3.68    1359.05
1         248.08       1.55     480.62
2           7.38      82.54  127338.31
3          15.50      22.33   12709.09


CHURN PREDICTION MODEL

In [21]:
last_90 = snapshot_date - pd.Timedelta(days=90)
active = df[df['InvoiceDate'] >= last_90]['CustomerID'].unique()
rfm['Churned'] = (~rfm['CustomerID'].isin(active)).astype(int)
churn_rate = rfm['Churned'].mean() * 100
print(f"Churn rate           : {churn_rate:.1f}%")
 
X_c = rfm[['Recency', 'Frequency', 'Monetary']].values
y_c = rfm['Churned'].values
X_tr, X_te, y_tr, y_te = train_test_split(X_c, y_c, test_size=0.2, random_state=42, stratify=y_c)
 
rf_model = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
rf_model.fit(X_tr, y_tr)
y_pred = rf_model.predict(X_te)
y_prob = rf_model.predict_proba(X_te)[:, 1]
auc = roc_auc_score(y_te, y_prob)
cv_auc = cross_val_score(rf_model, X_c, y_c, cv=5, scoring='roc_auc').mean()
 
print(f"Test AUC-ROC         : {auc:.4f}")
print(f"5-Fold CV AUC        : {cv_auc:.4f}")
print("\nClassification Report:")
print(classification_report(y_te, y_pred))
 
fi_series = pd.Series(rf_model.feature_importances_, index=['Recency', 'Frequency', 'Monetary'])

Churn rate           : 33.6%
Test AUC-ROC         : 1.0000
5-Fold CV AUC        : 1.0000

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       576
           1       1.00      1.00      1.00       292

    accuracy                           1.00       868
   macro avg       1.00      1.00      1.00       868
weighted avg       1.00      1.00      1.00       868



TIME SERIES & FORECASTING

In [22]:
monthly_rev = df.groupby('YearMonth')['Revenue'].sum().reset_index().sort_values('YearMonth')
X_t = np.arange(len(monthly_rev)).reshape(-1, 1)
y_t = monthly_rev['Revenue'].values
lr_model = LinearRegression().fit(X_t, y_t)
r2 = lr_model.score(X_t, y_t)
print(f"\nRevenue Trend R²     : {r2:.4f}")


Revenue Trend R²     : 0.3698


MARKET BASKET / ASSOCIATION ANALYSIS

In [23]:
top_products = df.groupby('Description')['Revenue'].sum().sort_values(ascending=False).head(20)
top_countries = df.groupby('Country')['Revenue'].sum().sort_values(ascending=False)

CORRELATION & STATISTICAL TESTS

In [24]:
r_fm, p_fm = pearsonr(rfm['Frequency'], rfm['Monetary'])
r_rf, p_rf = pearsonr(rfm['Recency'], rfm['Frequency'])
print(f"\nCorrelation Frequency↔Monetary : r={r_fm:.3f}, p={p_fm:.4f}")
print(f"Correlation Recency↔Frequency  : r={r_rf:.3f}, p={p_rf:.4f}")
 
# ANOVA: Monetary across segments
seg_groups = [rfm[rfm['Segment'] == s]['Monetary'].values for s in rfm['Segment'].unique()]
f_stat, p_anova = f_oneway(*seg_groups)
print(f"ANOVA (Monetary across segments): F={f_stat:.2f}, p={p_anova:.6f}")


Correlation Frequency↔Monetary : r=0.554, p=0.0000
Correlation Recency↔Frequency  : r=-0.261, p=0.0000
ANOVA (Monetary across segments): F=67.35, p=0.000000


Figures

In [27]:
plt.rcParams.update({'font.family': 'sans-serif', 'font.size': 11})
COLORS = ['#1565C0', '#4CAF50', '#FF9800', '#F44336', '#9C27B0', '#00BCD4', '#795548']
 
# ── Figure 1: RFM Distributions ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Figure 1 — RFM Metric Distributions\n(Online Retail II Dataset, n=4,338 customers)', fontsize=14, fontweight='bold')
for ax, col, color, xlabel in zip(axes, ['Recency', 'Frequency', 'Monetary'], COLORS[:3],
        ['Days Since Last Purchase', 'Number of Distinct Orders', 'Total Revenue Spent (£)']):
    data = rfm[col].clip(upper=rfm[col].quantile(0.98))
    ax.hist(data, bins=40, color=color, edgecolor='white', alpha=0.85)
    ax.set_title(f'{col} Distribution', fontweight='bold')
    ax.set_xlabel(xlabel); ax.set_ylabel('Number of Customers')
    ax.set_facecolor('#fafafa'); ax.grid(axis='y', alpha=0.3)
    mean_val = rfm[col].mean()
    ax.axvline(mean_val, color='red', linestyle='--', linewidth=1.5, label=f'Mean: {mean_val:.0f}')
    ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('figures/fig01_rfm_distributions.png', dpi=150, bbox_inches='tight')
plt.close()
print("✓ Figure 1 saved")
 
# ── Figure 2: RFM Segment Bar Chart ───────────────────────────────────────────
seg_counts = rfm['Segment'].value_counts()
SEG_COLORS = {'Champions':'#1565C0','Loyal Customers':'#4CAF50',
              'Potential Loyalists':'#2196F3','At Risk':'#FF9800','Lost':'#F44336'}
fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.bar(seg_counts.index, seg_counts.values,
              color=[SEG_COLORS[s] for s in seg_counts.index], edgecolor='white', linewidth=1.5, width=0.6)
for bar, val in zip(bars, seg_counts.values):
    pct = val / len(rfm) * 100
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 8,
            f'{val}\n({pct:.1f}%)', ha='center', fontsize=10, fontweight='bold')
ax.set_title('Figure 2 — Customer Segment Distribution (RFM Scoring)', fontsize=14, fontweight='bold')
ax.set_xlabel('Customer Segment', fontsize=12); ax.set_ylabel('Number of Customers', fontsize=12)
ax.set_facecolor('#fafafa'); ax.grid(axis='y', alpha=0.3); ax.set_ylim(0, seg_counts.max() * 1.2)
plt.tight_layout()
plt.savefig('figures/fig02_customer_segments_bar.png', dpi=150, bbox_inches='tight')
plt.close()
print("✓ Figure 2 saved")
 
# ── Figure 3: Elbow + Silhouette ──────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Figure 3 — Optimal K Selection for K-Means Clustering', fontsize=13, fontweight='bold')
ax1.plot(list(K_range), inertias, 'bo-', linewidth=2.2, markersize=9)
ax1.axvline(x=4, color='red', linestyle='--', linewidth=1.8, label='Optimal k=4')
ax1.set_title('Elbow Method (WCSS / Inertia)', fontweight='bold')
ax1.set_xlabel('Number of Clusters (k)'); ax1.set_ylabel('Within-Cluster Sum of Squares')
ax1.legend(); ax1.set_facecolor('#fafafa'); ax1.grid(alpha=0.3)
ax2.plot(list(K_range), sil_scores, 'gs-', linewidth=2.2, markersize=9)
best_k = list(K_range)[sil_scores.index(max(sil_scores))]
ax2.axvline(x=best_k, color='red', linestyle='--', linewidth=1.8, label=f'Best k={best_k}')
ax2.set_title('Silhouette Score by Cluster Count', fontweight='bold')
ax2.set_xlabel('Number of Clusters (k)'); ax2.set_ylabel('Average Silhouette Score')
ax2.legend(); ax2.set_facecolor('#fafafa'); ax2.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('figures/fig03_elbow_silhouette.png', dpi=150, bbox_inches='tight')
plt.close()
print("✓ Figure 3 saved")
 
# ── Figure 4: PCA 2D Cluster Scatter ──────────────────────────────────────────
cluster_colors_map = {0:'#1565C0', 1:'#4CAF50', 2:'#FF9800', 3:'#F44336'}
fig, ax = plt.subplots(figsize=(11, 7))
for c in range(4):
    mask = rfm['Cluster'] == c
    ax.scatter(rfm.loc[mask, 'PC1'], rfm.loc[mask, 'PC2'],
               label=f"Cluster {c}: {cluster_names[c]} (n={mask.sum()})",
               alpha=0.55, s=18, color=cluster_colors_map[c])
ax.set_title(f'Figure 4 — K-Means Customer Clusters (PCA 2D Projection)\nSilhouette Score = {final_sil:.4f}',
             fontsize=13, fontweight='bold')
ax.set_xlabel(f'PC1 — Explains {pca.explained_variance_ratio_[0]*100:.1f}% of Variance')
ax.set_ylabel(f'PC2 — Explains {pca.explained_variance_ratio_[1]*100:.1f}% of Variance')
ax.legend(markerscale=3, fontsize=10); ax.set_facecolor('#fafafa')
plt.tight_layout()
plt.savefig('figures/fig04_pca_cluster_scatter.png', dpi=150, bbox_inches='tight')
plt.close()
print("✓ Figure 4 saved")
 
# ── Figure 5: Cluster Profile Heatmap ─────────────────────────────────────────
cp = rfm.groupby('Cluster')[['Recency', 'Frequency', 'Monetary']].mean()
cp.index = [f"C{i}: {cluster_names[i]}" for i in cp.index]
cp_norm = (cp - cp.min()) / (cp.max() - cp.min())
fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(cp_norm, annot=cp.round(0), fmt='.0f', cmap='RdYlGn', ax=ax,
            linewidths=0.8, cbar_kws={'label': 'Normalized Score (0=Low, 1=High)'})
ax.set_title('Figure 5 — Cluster Profile Heatmap\n(Cell values = raw means; colors = normalized 0–1)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('RFM Dimensions'); ax.set_ylabel('Cluster')
plt.tight_layout()
plt.savefig('figures/fig05_cluster_heatmap.png', dpi=150, bbox_inches='tight')
plt.close()
print("✓ Figure 5 saved")
 
# ── Figure 6: Monthly Revenue Trend ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
x_pos = range(len(monthly_rev))
ax.fill_between(x_pos, monthly_rev['Revenue'], alpha=0.15, color='#1565C0')
ax.plot(x_pos, monthly_rev['Revenue'], 'o-', color='#1565C0', linewidth=2.5, markersize=7)
ax.set_xticks(list(x_pos))
ax.set_xticklabels(monthly_rev['YearMonth'], rotation=45, ha='right')
ax.set_title('Figure 6 — Monthly Revenue Trend (Dec 2010 – Dec 2011)', fontsize=13, fontweight='bold')
ax.set_xlabel('Month'); ax.set_ylabel('Total Revenue (£)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'£{x/1e6:.2f}M'))
ax.set_facecolor('#fafafa'); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('figures/fig06_monthly_revenue_trend.png', dpi=150, bbox_inches='tight')
plt.close()
print("✓ Figure 6 saved")
 
# ── Figure 7: Revenue Forecast ────────────────────────────────────────────────
n_future = 3
X_future = np.arange(len(monthly_rev) + n_future).reshape(-1, 1)
y_full_pred = lr_model.predict(X_future)
future_labels = list(monthly_rev['YearMonth']) + ['2012-01', '2012-02', '2012-03']
ci_width = np.std(y_t - lr_model.predict(X_t)) * 1.96
 
fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(range(len(monthly_rev)), y_t, 'o-', color='#1565C0', lw=2, label='Actual Revenue')
ax.plot(range(len(monthly_rev) + n_future), y_full_pred, '--', color='#F44336', lw=2, label=f'Linear Trend (R²={r2:.3f})')
ax.fill_between(range(len(monthly_rev), len(monthly_rev) + n_future),
                y_full_pred[len(monthly_rev):] - ci_width,
                y_full_pred[len(monthly_rev):] + ci_width,
                alpha=0.25, color='#F44336', label='95% Prediction Interval')
ax.axvline(x=len(monthly_rev) - 1, color='gray', linestyle=':', linewidth=1.5, label='Forecast Start')
ax.set_xticks(range(len(future_labels)))
ax.set_xticklabels(future_labels, rotation=45, ha='right')
ax.set_title('Figure 7 — Revenue Forecast: 3-Month Linear Trend Extrapolation', fontsize=13, fontweight='bold')
ax.set_ylabel('Total Revenue (£)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'£{x/1e6:.2f}M'))
ax.legend(); ax.set_facecolor('#fafafa'); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('figures/fig07_revenue_forecast.png', dpi=150, bbox_inches='tight')
plt.close()
print("✓ Figure 7 saved")
 
# ── Figure 8: Top 10 Countries ────────────────────────────────────────────────
top10 = top_countries.head(10)
fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(top10.index[::-1], top10.values[::-1],
               color=sns.color_palette('Blues_d', 10), edgecolor='white')
for bar, val in zip(bars, top10.values[::-1]):
    ax.text(bar.get_width() + 5000, bar.get_y() + bar.get_height()/2,
            f'£{val:,.0f}', va='center', fontsize=9, fontweight='bold')
ax.set_title('Figure 8 — Top 10 Countries by Total Revenue', fontsize=13, fontweight='bold')
ax.set_xlabel('Total Revenue (£)')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'£{x/1e6:.1f}M'))
ax.set_facecolor('#fafafa')
plt.tight_layout()
plt.savefig('figures/fig08_top_countries.png', dpi=150, bbox_inches='tight')
plt.close()
print("✓ Figure 8 saved")
 
# ── Figure 9: RFM Correlation Heatmap ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
corr_m = rfm[['Recency', 'Frequency', 'Monetary']].corr()
mask = np.zeros_like(corr_m, dtype=bool)
sns.heatmap(corr_m, annot=True, fmt='.3f', cmap='coolwarm', vmin=-1, vmax=1, ax=ax,
            linewidths=1, square=True, cbar_kws={'label': 'Pearson r'})
ax.set_title('Figure 9 — RFM Pearson Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/fig09_rfm_correlation.png', dpi=150, bbox_inches='tight')
plt.close()
print("✓ Figure 9 saved")
 
# ── Figure 10: Revenue by Segment Pie ─────────────────────────────────────────
seg_rev = rfm.groupby('Segment')['Monetary'].sum().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(9, 7))
wedges, texts, autotexts = ax.pie(
    seg_rev.values, labels=seg_rev.index, autopct='%1.1f%%',
    colors=[SEG_COLORS[s] for s in seg_rev.index], startangle=140,
    pctdistance=0.82, explode=[0.04] + [0] * (len(seg_rev) - 1))
for t in autotexts: t.set_fontsize(10); t.set_fontweight('bold')
ax.set_title('Figure 10 — Revenue Contribution by Customer Segment', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/fig10_segment_revenue_pie.png', dpi=150, bbox_inches='tight')
plt.close()
print("✓ Figure 10 saved")
 
# ── Figure 11: CLV by Segment ─────────────────────────────────────────────────
clv_seg = rfm.groupby('Segment')['CLV_Estimated'].mean().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.bar(clv_seg.index, clv_seg.values,
              color=[SEG_COLORS[s] for s in clv_seg.index], edgecolor='white', width=0.6)
for bar, val in zip(bars, clv_seg.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'£{val:,.0f}', ha='center', fontsize=10, fontweight='bold')
ax.set_title('Figure 11 — Estimated Customer Lifetime Value (CLV) by Segment\n(2-Year Projection)', fontsize=13, fontweight='bold')
ax.set_ylabel('Average Estimated CLV (£)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
ax.set_facecolor('#fafafa'); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('figures/fig11_clv_by_segment.png', dpi=150, bbox_inches='tight')
plt.close()
print("✓ Figure 11 saved")

# ── Figure 12: Day-of-Week Pattern ────────────────────────────────────────────
dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_rev = df.groupby('DayOfWeek')['Revenue'].sum().reindex(dow_order)
dow_orders = df.groupby('DayOfWeek')['InvoiceNo'].nunique().reindex(dow_order)
fig, ax1 = plt.subplots(figsize=(12, 5))
bars = ax1.bar(dow_order, dow_rev.values, color='#2196F3', alpha=0.75, edgecolor='white')
ax1.set_ylabel('Total Revenue (£)', color='#1565C0', fontsize=11)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'£{x/1e6:.1f}M'))
ax1.set_title('Figure 12 — Revenue and Order Volume by Day of Week', fontsize=13, fontweight='bold')
ax2 = ax1.twinx()
ax2.plot(dow_order, dow_orders.values, 'ro-', linewidth=2.2, markersize=9, label='Unique Orders')
ax2.set_ylabel('Number of Unique Orders', color='red', fontsize=11)
ax1.set_facecolor('#fafafa')
lines, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines, labels2, loc='upper left')
plt.tight_layout()
plt.savefig('figures/fig12_dow_pattern.png', dpi=150, bbox_inches='tight')
plt.close()
print("✓ Figure 12 saved")
 
# ── Figure 13: Top 15 Products ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 8))
tp15 = top_products.head(15)
ax.barh(range(len(tp15)), tp15.values[::-1], color=sns.color_palette('viridis', 15), edgecolor='white')
ax.set_yticks(range(len(tp15)))
ax.set_yticklabels([d[:45] for d in tp15.index[::-1]], fontsize=9)
ax.set_title('Figure 13 — Top 15 Products by Total Revenue', fontsize=13, fontweight='bold')
ax.set_xlabel('Total Revenue (£)')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
ax.set_facecolor('#fafafa')
plt.tight_layout()
plt.savefig('figures/fig13_top_products.png', dpi=150, bbox_inches='tight')
plt.close()
print("✓ Figure 13 saved")
 
# ── Figure 14: ROC Curve ──────────────────────────────────────────────────────
fpr, tpr, _ = roc_curve(y_te, y_prob)
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr, tpr, color='#1565C0', lw=2.5, label=f'RF Churn Model (AUC = {auc:.4f})')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random Baseline (AUC = 0.50)')
ax.fill_between(fpr, tpr, alpha=0.12, color='#1565C0')
ax.set_xlabel('False Positive Rate (1 – Specificity)', fontsize=12)
ax.set_ylabel('True Positive Rate (Sensitivity)', fontsize=12)
ax.set_title('Figure 14 — Churn Prediction ROC Curve\n(Random Forest, 5-Fold CV AUC = {:.4f})'.format(cv_auc),
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11); ax.set_facecolor('#fafafa')
plt.tight_layout()
plt.savefig('figures/fig14_roc_curve.png', dpi=150, bbox_inches='tight')
plt.close()
print("✓ Figure 14 saved")
 
# ── Figure 15: Feature Importance ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
fi_sorted = fi_series.sort_values()
colors_fi = ['#F44336' if v < 0.3 else '#FF9800' if v < 0.4 else '#4CAF50' for v in fi_sorted.values]
fi_sorted.plot.barh(ax=ax, color=colors_fi, edgecolor='white')
for i, v in enumerate(fi_sorted.values):
    ax.text(v + 0.003, i, f'{v:.3f} ({v*100:.1f}%)', va='center', fontsize=11, fontweight='bold')
ax.set_title('Figure 15 — Churn Model Feature Importance (Random Forest)', fontsize=13, fontweight='bold')
ax.set_xlabel('Gini Importance Score'); ax.set_xlim(0, 0.75); ax.set_facecolor('#fafafa')
plt.tight_layout()
plt.savefig('figures/fig15_feature_importance.png', dpi=150, bbox_inches='tight')
plt.close()
print("✓ Figure 15 saved")
 
# ── Figure 16: Confusion Matrix ───────────────────────────────────────────────
cm = confusion_matrix(y_te, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, linewidths=0.5,
            xticklabels=['Predicted: Active', 'Predicted: Churned'],
            yticklabels=['Actual: Active', 'Actual: Churned'])
ax.set_title('Figure 16 — Churn Prediction Confusion Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/fig16_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.close()
print("✓ Figure 16 saved")
 
# ── Figure 17: Recency vs Monetary Scatter by Segment ─────────────────────────
fig, ax = plt.subplots(figsize=(11, 7))
for seg, grp in rfm.groupby('Segment'):
    ax.scatter(grp['Recency'], grp['Monetary'].clip(upper=grp['Monetary'].quantile(0.95)),
               label=seg, alpha=0.5, s=18, color=SEG_COLORS.get(seg, 'gray'))
ax.set_title('Figure 17 — Recency vs Monetary Value by Customer Segment', fontsize=13, fontweight='bold')
ax.set_xlabel('Recency (Days Since Last Purchase)')
ax.set_ylabel('Monetary Value (£, 95th pct. capped)')
ax.legend(markerscale=3, fontsize=10); ax.set_facecolor('#fafafa')
plt.tight_layout()
plt.savefig('figures/fig17_rfm_scatter.png', dpi=150, bbox_inches='tight')
plt.close()
print("✓ Figure 17 saved")
 
# ── Figure 18: New vs Returning Customers Monthly ─────────────────────────────
df['FirstPurchaseMonth'] = df.groupby('CustomerID')['InvoiceDate'].transform('min').dt.strftime('%Y-%m')
df['IsNew'] = df['YearMonth'] == df['FirstPurchaseMonth']
all_months = sorted(df['YearMonth'].unique())
new_monthly = df[df['IsNew']].groupby('YearMonth')['CustomerID'].nunique().reindex(all_months, fill_value=0)
ret_monthly = df[~df['IsNew']].groupby('YearMonth')['CustomerID'].nunique().reindex(all_months, fill_value=0)
fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(all_months, new_monthly.values, label='New Customers', color='#4CAF50', alpha=0.85, edgecolor='white')
ax.bar(all_months, ret_monthly.values, bottom=new_monthly.values,
       label='Returning Customers', color='#2196F3', alpha=0.85, edgecolor='white')
ax.set_xticks(range(len(all_months)))
ax.set_xticklabels(all_months, rotation=45, ha='right')
ax.set_title('Figure 18 — Monthly New vs Returning Customer Counts', fontsize=13, fontweight='bold')
ax.set_ylabel('Customer Count'); ax.legend(fontsize=11); ax.set_facecolor('#fafafa')
plt.tight_layout()
plt.savefig('figures/fig18_new_vs_returning.png', dpi=150, bbox_inches='tight')
plt.close()
print("✓ Figure 18 saved")
 
# ── Figure 19: Monetary Boxplot by Cluster ────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
data_bp = [rfm[rfm['Cluster'] == c]['Monetary'].clip(upper=rfm['Monetary'].quantile(0.95)).values for c in range(4)]
bp = ax.boxplot(data_bp, patch_artist=True, notch=False,
                labels=[f"C{c}: {cluster_names[c]}" for c in range(4)])
for patch, color in zip(bp['boxes'], [cluster_colors_map[c] for c in range(4)]):
    patch.set_facecolor(color); patch.set_alpha(0.7)
ax.set_title('Figure 19 — Monetary Value Distribution by Cluster\n(Capped at 95th percentile)', fontsize=13, fontweight='bold')
ax.set_ylabel('Monetary Value (£)'); ax.set_facecolor('#fafafa')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
plt.tight_layout()
plt.savefig('figures/fig19_cluster_boxplot.png', dpi=150, bbox_inches='tight')
plt.close()
print("✓ Figure 19 saved")
 
# ── Figure 20: Cohort Retention Heatmap ───────────────────────────────────────
df['CohortMonth'] = df.groupby('CustomerID')['InvoiceDate'].transform('min').dt.to_period('M')
df['OrderMonth'] = df['InvoiceDate'].dt.to_period('M')
df['CohortIndex'] = (df['OrderMonth'] - df['CohortMonth']).apply(lambda x: x.n)
cohort_data = df.groupby(['CohortMonth', 'CohortIndex'])['CustomerID'].nunique().reset_index()
cohort_pivot = cohort_data.pivot_table(index='CohortMonth', columns='CohortIndex', values='CustomerID')
cohort_size = cohort_pivot[0]
retention = (cohort_pivot.divide(cohort_size, axis=0) * 100).round(1)
fig, ax = plt.subplots(figsize=(15, 8))
sns.heatmap(retention.iloc[:12, :9], annot=True, fmt='.0f', cmap='YlOrRd_r',
            ax=ax, linewidths=0.4, cbar_kws={'label': 'Retention Rate (%)'})
ax.set_title('Figure 20 — Customer Cohort Retention Matrix (%)\n(Row = Acquisition Month, Column = Months Since First Purchase)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Months Since First Purchase'); ax.set_ylabel('Acquisition Cohort (Month)')
plt.tight_layout()
plt.savefig('figures/fig20_cohort_retention.png', dpi=150, bbox_inches='tight')
plt.close()
print("✓ Figure 20 saved")
 

✓ Figure 1 saved
✓ Figure 2 saved
✓ Figure 3 saved
✓ Figure 4 saved
✓ Figure 5 saved
✓ Figure 6 saved
✓ Figure 7 saved
✓ Figure 8 saved
✓ Figure 9 saved
✓ Figure 10 saved
✓ Figure 11 saved
✓ Figure 12 saved
✓ Figure 13 saved
✓ Figure 14 saved
✓ Figure 15 saved
✓ Figure 16 saved
✓ Figure 17 saved
✓ Figure 18 saved
✓ Figure 19 saved
✓ Figure 20 saved


Summary

In [28]:
print(f"""
KEY MODEL STATISTICS (for answering the 200 questions):
─────────────────────────────────────────────────────────
Dataset shape (clean)  : {df.shape[0]:,} transactions, 4,338 customers
Date range             : Dec 2010 – Dec 2011
Countries              : 37
Total revenue          : £{df['Revenue'].sum():,.0f}
 
RFM (mean)             : Recency={rfm['Recency'].mean():.0f}d  Frequency={rfm['Frequency'].mean():.1f} orders  Monetary=£{rfm['Monetary'].mean():,.0f}
RFM (median)           : Recency={rfm['Recency'].median():.0f}d  Frequency={rfm['Frequency'].median():.0f} orders  Monetary=£{rfm['Monetary'].median():,.0f}
RFM (std)              : Recency={rfm['Recency'].std():.0f}d  Frequency={rfm['Frequency'].std():.1f}  Monetary=£{rfm['Monetary'].std():,.0f}
 
Segments               : Champions=1268  Loyal=843  Potential=936  AtRisk=988  Lost=303
K-Means clusters (k=4) : Silhouette={final_sil:.4f}
Churn rate             : {rfm['Churned'].mean()*100:.1f}%
Churn model AUC-ROC    : {auc:.4f}  (5-Fold CV: {cv_auc:.4f})
Revenue trend R²       : {r2:.4f}
Top feature (churn)    : Recency ({fi_series['Recency']*100:.1f}% importance)
 
Correlation F↔M        : r={r_fm:.3f}  p={p_fm:.4f}
ANOVA (Monetary/Seg)   : F={f_stat:.2f}  p={p_anova:.6e}
""")


KEY MODEL STATISTICS (for answering the 200 questions):
─────────────────────────────────────────────────────────
Dataset shape (clean)  : 397,884 transactions, 4,338 customers
Date range             : Dec 2010 – Dec 2011
Countries              : 37
Total revenue          : £8,911,408
 
RFM (mean)             : Recency=93d  Frequency=4.3 orders  Monetary=£2,054
RFM (median)           : Recency=51d  Frequency=2 orders  Monetary=£674
RFM (std)              : Recency=100d  Frequency=7.7  Monetary=£8,989
 
Segments               : Champions=1268  Loyal=843  Potential=936  AtRisk=988  Lost=303
K-Means clusters (k=4) : Silhouette=0.6162
Churn rate             : 33.6%
Churn model AUC-ROC    : 1.0000  (5-Fold CV: 1.0000)
Revenue trend R²       : 0.3698
Top feature (churn)    : Recency (87.7% importance)
 
Correlation F↔M        : r=0.554  p=0.0000
ANOVA (Monetary/Seg)   : F=67.35  p=2.261030e-55

